# Notebook 5: From Model to Production API

**Series:** Fine-Tuning Flan-T5 for Dialogue Summarization (5 of 5)

---

## What this notebook covers

This is the **production layer** notebook. Everything you learned in notebooks 1-4 — loading data, tokenizing sequences, configuring training, running inference — gets wrapped into a real HTTP API here.

By the end of this notebook you will:

- Understand the three files in `inference/` and how they fit together
- Read and explain every line of `inference/app.py` and `inference/logger.py`
- Know why FastAPI's `lifespan` context manager replaced the old `@app.on_event` pattern
- Run the API **inside this notebook** using `TestClient` (no subprocess, no ports)
- Call `/health` and `/summarize` and inspect the responses
- Understand how W&B Weave traces every inference call
- See how the whole pipeline maps to a live AWS App Runner deployment

> **No GPU required.** The model is mocked in this notebook so everything runs on CPU in under a second — including on Google Colab's free tier.

In [ ]:
!pip install -q "fastapi>=0.133.1" "uvicorn[standard]>=0.41.0" "pydantic>=2.12.5" \
    "torch>=2.10.0" "transformers>=5.2.0" "httpx>=0.27.0"

## The `inference/` package: three files, one job

The production service lives in the `inference/` directory. There are exactly three source files, each with a single responsibility:

| File | Responsibility |
|------|----------------|
| `model.py` | Loads the Flan-T5 model once at startup; exposes `predict(text)` |
| `app.py` | FastAPI routes that wrap `predict()` behind HTTP endpoints |
| `logger.py` | Logs every query/response pair to W&B Weave for observability |

Here is how a single request flows through the system:

```
                        ┌─────────────────────────────────────┐
                        │           inference/app.py           │
                        │                                     │
 HTTP POST /summarize → │  SummarizeRequest (Pydantic)        │
                        │       │                             │
                        │       ▼                             │
                        │  predict(text)  ←── model.py        │
                        │       │                             │
                        │       ▼                             │
                        │  SummarizeResponse (Pydantic) ──────┼──→ HTTP 200 JSON
                        │       │                             │
                        │       ▼ (side branch)               │
                        │  log_inference()  ←── logger.py     │
                        │       │                             │
                        └───────┼─────────────────────────────┘
                                ▼
                        W&B Weave dashboard
                        (trace stored for analysis)
```

The `logger.py` side branch is intentional: logging is a side effect, not part of the response path. If Weave is unavailable the response still succeeds.

In [ ]:
# ── Cell 4: Pydantic models ────────────────────────────────────────────────
#
# FastAPI uses Pydantic to validate incoming JSON and serialize outgoing JSON.
# These two models define the exact shape of a /summarize request and response.

from pydantic import BaseModel, ConfigDict

class SummarizeRequest(BaseModel):
    # ConfigDict(strict=False) means Pydantic will coerce compatible types.
    # For example, an integer "128" sent as a string will be accepted as 128.
    model_config = ConfigDict(strict=False)

    text: str          # required — no default
    max_length: int = 128  # optional — defaults to 128 tokens

class SummarizeResponse(BaseModel):
    summary: str       # the only field returned to the caller

# ── Demonstrate validation ─────────────────────────────────────────────────

# Happy path: just the required field
req = SummarizeRequest(text="Amanda baked cookies.")
print("Request:", req)
print("text field:", req.text)
print("max_length default:", req.max_length)

# JSON serialization — this is what FastAPI sends back over the wire
import json
print("As JSON:", req.model_dump_json())

# Override the default
req2 = SummarizeRequest(text="Long document...", max_length=64)
print("\nWith custom max_length:", req2.model_dump_json())

# Validation error — missing the required field
from pydantic import ValidationError
try:
    bad = SummarizeRequest()  # 'text' is required
except ValidationError as e:
    print("\nValidation error (expected):", e.error_count(), "error(s)")

In [ ]:
# ── Cell 5: The lifespan context manager ──────────────────────────────────
#
# FastAPI used to use @app.on_event("startup") / @app.on_event("shutdown").
# As of FastAPI 0.93, that API is deprecated in favor of the `lifespan`
# context manager — a standard Python asynccontextmanager that covers both
# startup (code before `yield`) and shutdown (code after `yield`) in one place.
#
# Why it matters: the model is ~900 MB. You want to load it ONCE when the
# server starts, store it in app.state, and reuse it for every request.
# Loading inside the route handler would re-load on every call — very slow.

from contextlib import asynccontextmanager
from fastapi import FastAPI

@asynccontextmanager
async def lifespan(app):
    # ── Everything BEFORE yield runs at startup ────────────────────────────
    print("▶ Startup: loading resources...")
    app.state.data = {"loaded": True}  # simulated resource (real code: load_model())
    yield
    # ── Everything AFTER yield runs at shutdown ────────────────────────────
    print("■ Shutdown: releasing resources...")

demo_app = FastAPI(lifespan=lifespan)
print("App created with lifespan. In production, startup runs on first request.")

# In inference/app.py the real lifespan looks like:
#
#   @asynccontextmanager
#   async def lifespan(app: FastAPI):
#       app.state.model = load_model()   <-- heavy work done here, once
#       yield
#
# The route handler then accesses app.state.model without reloading it.

In [ ]:
# ── Cell 6: Full app definition with mocked model ─────────────────────────
#
# This is a self-contained replica of inference/app.py.
# The only difference: instead of calling inference.model.predict() (which
# would download ~900 MB), we use mock_predict() that returns a plausible
# string immediately. The API structure, routing, and Pydantic models are
# identical to production.

import torch
from unittest.mock import MagicMock, patch
from contextlib import asynccontextmanager
from fastapi import FastAPI
from pydantic import BaseModel, ConfigDict


class SummarizeRequest(BaseModel):
    model_config = ConfigDict(strict=False)
    text: str
    max_length: int = 128

class SummarizeResponse(BaseModel):
    summary: str


# ── Mock model ─────────────────────────────────────────────────────────────
# In production, inference/model.py loads google/flan-t5-base (or a
# fine-tuned checkpoint) and runs it through the HuggingFace pipeline.
# Here we return a deterministic placeholder so no download is needed.

def mock_predict(text, max_new_tokens=128):
    return f"[MOCK SUMMARY] Condensed version of: {text[:50]}..."


# ── Lifespan ───────────────────────────────────────────────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.ready = True   # production: app.state.model = load_model()
    yield


# ── Application ────────────────────────────────────────────────────────────
app = FastAPI(title="Flan-T5 Summarizer API", version="1.0.0", lifespan=lifespan)


@app.get("/health")
async def health():
    """Liveness probe — used by AWS App Runner and load balancers."""
    return {"status": "ok"}


@app.post("/summarize", response_model=SummarizeResponse)
async def summarize(request: SummarizeRequest):
    """
    Main inference endpoint.

    FastAPI automatically:
      - parses the JSON body into SummarizeRequest
      - validates required fields
      - serializes the return value through SummarizeResponse
      - returns HTTP 422 if validation fails
    """
    summary = mock_predict(request.text, max_new_tokens=request.max_length)
    # In production this line also appears:
    #   log_inference(query=request.text, response=summary)
    return SummarizeResponse(summary=summary)


print("App defined.")
print(f"Routes: {[r.path for r in app.routes]}")

In [ ]:
# ── Cell 7: Call the API with TestClient ──────────────────────────────────
#
# fastapi.testclient.TestClient wraps the ASGI app in a requests-compatible
# interface. No server process, no port binding — it calls the app in-process.
# This is the same approach used in the project's test suite (tests/).
#
# Using `with TestClient(app) as client:` triggers the lifespan context
# manager: startup runs on __enter__, shutdown on __exit__.

from fastapi.testclient import TestClient

with TestClient(app) as client:

    # ── 1. Health check ────────────────────────────────────────────────────
    health_resp = client.get("/health")
    print("GET /health")
    print(f"  Status : {health_resp.status_code}")
    print(f"  Body   : {health_resp.json()}")
    print()

    # ── 2. Summarize with default max_length ───────────────────────────────
    payload = {
        "text": (
            "Alex: Did you finish the report? "
            "Jordan: Almost done, just adding charts. "
            "Alex: Need it by 5pm. "
            "Jordan: Will have it by 3pm."
        ),
        "max_length": 64,
    }
    summ_resp = client.post("/summarize", json=payload)
    print("POST /summarize")
    print(f"  Status : {summ_resp.status_code}")
    print(f"  Body   : {summ_resp.json()}")
    print()

    # ── 3. Validation error — missing required field ───────────────────────
    bad_resp = client.post("/summarize", json={"max_length": 32})  # no 'text'
    print("POST /summarize (missing 'text' field)")
    print(f"  Status : {bad_resp.status_code}  ← FastAPI returns 422 Unprocessable Entity")
    detail = bad_resp.json().get("detail", [])
    if detail:
        print(f"  Error  : {detail[0]['msg']} — field: {detail[0]['loc']}")

## Observability: `inference/logger.py` and W&B Weave

Once an API is in production you need answers to questions like:

- What inputs did the model receive?
- What did it return?
- Are there patterns in the inputs that cause bad summaries?
- Is latency increasing over time?

The `inference/logger.py` file answers all of these using **W&B Weave** — a lightweight tracing layer built on top of Weights & Biases.

Here is the full content of `inference/logger.py`:

```python
"""W&B Weave observability for inference logging."""
import os
import weave

_initialized = False

def _ensure_init():
    global _initialized
    if not _initialized:
        project = os.environ.get("WANDB_PROJECT", "flan-t5-summarizer")
        weave.init(project_name=project)
        _initialized = True

@weave.op()
def log_inference(query: str, response: str) -> dict:
    _ensure_init()
    return {"query": query, "response": response}
```

**Key ideas:**

- `@weave.op()` is a decorator that automatically captures the function's **inputs and outputs** as a trace. You do not have to write any serialization code — Weave handles it.
- Every call to `log_inference()` creates one trace entry in the **W&B Weave dashboard**, where you can filter, sort, and replay traces.
- `_ensure_init()` uses a module-level flag so `weave.init()` is called **exactly once** per process, not once per request.
- Two environment variables control where traces are stored:
  - `WANDB_API_KEY` — your W&B credentials (set in GitHub Actions secrets or `.env`)
  - `WANDB_PROJECT` — the W&B project name (defaults to `"flan-t5-summarizer"`)

If either variable is missing, `log_inference()` will raise an error on first call. In production, these are injected as App Runner environment variables.

In [ ]:
# ── Cell 9: How @weave.op() works — conceptual illustration ───────────────
#
# We do NOT call weave.init() here (that would require WANDB_API_KEY).
# Instead, we build a simplified version of @weave.op() from scratch to show
# exactly what the decorator does: intercept the call, record inputs,
# run the function, record the output.

def weave_op_simplified(func):
    """
    Simplified version of @weave.op() to illustrate the concept.

    The real @weave.op():
      - Sends inputs/outputs to the W&B Weave backend over HTTP
      - Records wall-clock time, exception info, and call hierarchy
      - Is async-aware (works with `async def` functions too)

    This version just prints to stdout so you can see the interception.
    """
    def wrapper(*args, **kwargs):
        print(f"[TRACE] Calling '{func.__name__}'")
        # Show a truncated view of inputs (real Weave stores everything)
        if args:
            print(f"[TRACE] args[0] = {str(args[0])[:60]!r}")
        for k, v in kwargs.items():
            print(f"[TRACE] kwarg {k!r} = {str(v)[:60]!r}")

        result = func(*args, **kwargs)  # call the actual function

        print(f"[TRACE] Output = {str(result)[:60]!r}")
        print(f"[TRACE] → stored in W&B project 'flan-t5-summarizer'")
        return result
    return wrapper


# Replicate inference/logger.py with our simplified decorator
@weave_op_simplified
def log_inference(query: str, response: str) -> dict:
    """Stores one inference record. In production, @weave.op() replaces the decorator above."""
    return {"query": query, "response": response}


# Demonstrate — this is what app.py calls after every /summarize request
print("=" * 60)
result = log_inference(
    query="Alex: Need report by 5pm. Jordan: Will have it by 3pm.",
    response="Jordan will have the report ready by 3pm.",
)
print("=" * 60)
print("Returned dict:", result)

## From notebook to production deployment

Everything you have seen in this notebook corresponds directly to the live production system. Here is the mapping:

### Model loading
In production, `inference/model.py` loads **`google/flan-t5-base`** by default. If you set the `HF_MODEL_ID` environment variable, it loads that checkpoint from the HuggingFace Hub instead — so a fine-tuned model pushed in notebook 4 can be served here without changing any code:

```bash
HF_MODEL_ID=your-username/flan-t5-dialogue-summarizer
```

### Containerization
The `inference/Dockerfile` packages the FastAPI app into a Docker image. It installs dependencies from `pyproject.toml`, copies the `inference/` package, and runs:

```
uvicorn inference.app:app --host 0.0.0.0 --port 8080
```

### CI/CD
`.github/workflows/deploy.yml` triggers on every push to `main`. It:
1. Builds the Docker image
2. Pushes it to Amazon ECR
3. Deploys to **AWS App Runner**, which handles auto-scaling and HTTPS

### Observability
Once deployed, every call to `POST /summarize` calls `log_inference()`, which traces inputs and outputs to the **W&B Weave** dashboard. You can view all traces at `https://wandb.ai/<your-entity>/flan-t5-summarizer/weave`.

---

## The complete pipeline

```
Notebook 1  →  Raw data (SAMSum dialogue dataset)
Notebook 2  →  Tokenization (input_ids, attention_mask, labels)
Notebook 3  →  Training loop (Seq2SeqTrainer, PEFT/LoRA config)
Notebook 4  →  Inference (generate(), beam search, decoding)
Notebook 5  →  API + Observability  ← you are here
```

You have now seen the entire pipeline: **data → tokenization → training → inference → API → observability**.

The model you trained in notebook 3, pushed to HuggingFace Hub in notebook 4, is now being served at scale behind a REST API with full tracing of every request.